In [1]:
import polars as pl 
from dbconfig import engine
print('Environment ready!')

Environment ready!


In [2]:
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_float_precision(2)

polars.config.Config

In [3]:
# constants
INITIAL_MONTHLY = 5000
SAVINGS_RATE = 0.0331
RD_RATE = 0.0585
NIFTY_50 = 0.1185
INFLATION = 0.0494

In [4]:
def simulate_benchmark(
    years: int,
    annual_rate: float,
    contribution_growth: float,
) -> pl.DataFrame:

    monthly_rate = annual_rate / 12

    monthly_contribution = INITIAL_MONTHLY

    balance = 0.0
    total_contributions = 0.0
    total_real_contributions = 0.0
    total_interest = 0.0

    month_number = 0

    rows = []

    for year in range(1, years + 1):

        for _ in range(12):

            month_number += 1

            monthly_inflation_factor = (
                (1 + INFLATION) ** (month_number / 12)
            )

            balance += monthly_contribution
            total_contributions += monthly_contribution

            total_real_contributions += (
                monthly_contribution / monthly_inflation_factor
            )

            interest = balance * monthly_rate

            balance += interest
            total_interest += interest

        inflation_factor = (1 + INFLATION) ** year
        real_balance = balance / inflation_factor
        purchasing_power_retained = (real_balance / total_real_contributions)

        rows.append({
            "year": year,
            "monthly_contribution": round(monthly_contribution, 2),
            "total_contributions": round(total_contributions, 2),
            "real_contributions": round(total_real_contributions, 2),
            "interest_earned": round(total_interest, 2),
            "nominal_corpus": round(balance, 2),
            "inflation_factor": round(inflation_factor, 4),
            "real_corpus": round(real_balance, 2),
            "real_wealth_created": round(
                real_balance - total_real_contributions, 2),
            "purchasing_power_retained": round(
        purchasing_power_retained * 100, 2),
        })

        monthly_contribution *= (
            1 + INFLATION * contribution_growth
        )

    return pl.DataFrame(rows)

In [5]:
savings_fixed = simulate_benchmark(
    years=30,
    annual_rate=SAVINGS_RATE,
    contribution_growth=0.0,
)

In [6]:
savings_half = simulate_benchmark(
    years=30,
    annual_rate=SAVINGS_RATE,
    contribution_growth=0.5,
)

In [7]:
savings_full = simulate_benchmark(
    years=30,
    annual_rate=SAVINGS_RATE,
    contribution_growth=1.0,
)

In [8]:
rd = simulate_benchmark(
    years=10,
    annual_rate=RD_RATE,
    contribution_growth=0.0,
)

In [14]:
nifty_50_fixed = simulate_benchmark(
        years = 30,
        annual_rate = NIFTY_50,
        contribution_growth = 0.0
        )

In [12]:
nifty_50_half = simulate_benchmark(
        years = 30,
        annual_rate = NIFTY_50,
        contribution_growth = 0.5
        )

In [13]:
nifty_50_full = simulate_benchmark(
        years = 30,
        annual_rate = NIFTY_50,
        contribution_growth = 1.0
        )

In [15]:
summary = pl.DataFrame([
    {
        "scenario": "Savings - Fixed",
        **savings_fixed.tail(1).to_dicts()[0]
    },
    {
        "scenario": "Savings - Half Inflation",
        **savings_half.tail(1).to_dicts()[0]
    },
    {
        "scenario": "Savings - Full Inflation",
        **savings_full.tail(1).to_dicts()[0]
    },
    {
        "scenario": "Recurring Deposit",
        **rd.tail(1).to_dicts()[0]
    },
    {
        "scenario": "Nifty 50 - Fixed",
        **nifty_50_fixed.tail(1).to_dicts()[0]
    },
    {
        "scenario": "Nifty 50 - Half Inflation",
        **nifty_50_half.tail(1).to_dicts()[0]
    },
    {
        "scenario": "Nifty 50 - Full Inflation",
        **nifty_50_full.tail(1).to_dicts()[0]
    },
])


In [16]:
summary

scenario,year,monthly_contribution,total_contributions,real_contributions,interest_earned,nominal_corpus,inflation_factor,real_corpus,real_wealth_created,purchasing_power_retained
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Savings - Fixed""",30,5000.00,1800000.00,949533.16,1282132.88,3082132.88,4.25,725470.17,-224062.99,76.40
"""Savings - Half Inflation""",30,10145.54,2621606.14,1268144.70,1605966.98,4227573.12,4.25,995083.06,-273061.64,78.47
"""Savings - Full Inflation""",30,20242.34,3945501.09,1753764.24,2060707.51,6006208.60,4.25,1413736.97,-340027.27,80.61
"""Recurring Deposit""",10,5000.00,600000.00,475086.05,216722.76,816722.76,1.62,504271.08,29185.03,106.14
"""Nifty 50 - Fixed""",30,5000.00,1800000.00,949533.16,15269693.45,17069693.45,4.25,4017851.91,3068318.75,423.14
"""Nifty 50 - Half Inflation""",30,10145.54,2621606.14,1268144.70,17957771.18,20579377.32,4.25,4843958.73,3575814.03,381.97
"""Nifty 50 - Full Inflation""",30,20242.34,3945501.09,1753764.24,21511489.05,25456990.14,4.25,5992047.65,4238283.41,341.67


In [11]:
savings_fixed.head(10)

year,monthly_contribution,total_contributions,real_contributions,interest_earned,nominal_corpus,inflation_factor,real_corpus,real_wealth_created,purchasing_power_retained
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,5000.00,60000.00,58458.81,1086.71,61086.71,1.05,58211.08,-247.73,99.58
2,5000.00,120000.00,114165.70,4226.34,124226.34,1.10,112805.84,-1359.86,98.81
3,5000.00,180000.00,167250.21,9487.89,189487.89,1.16,163967.69,-3282.51,98.04
4,5000.00,240000.00,217835.79,16942.68,256942.68,1.21,211871.24,-5964.55,97.26
5,5000.00,300000.00,266040.09,26664.41,326664.41,1.27,256682.65,-9357.43,96.48
6,5000.00,360000.00,311975.19,38729.26,398729.26,1.34,298560.09,-13415.10,95.70
7,5000.00,420000.00,355747.91,53215.98,473215.98,1.40,337654.07,-18093.85,94.91
8,5000.00,480000.00,397460.06,70205.96,550205.96,1.47,374107.84,-23352.22,94.12
9,5000.00,540000.00,437208.63,89783.33,629783.33,1.54,408057.74,-29150.88,93.33


In [12]:
savings_half.head(10)

year,monthly_contribution,total_contributions,real_contributions,interest_earned,nominal_corpus,inflation_factor,real_corpus,real_wealth_created,purchasing_power_retained
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,5000.00,60000.00,58458.81,1086.71,61086.71,1.05,58211.08,-247.73,99.58
2,5123.50,121482.00,115541.66,4253.18,125735.18,1.10,114175.97,-1365.69,98.82
3,5250.05,184482.61,171280.93,9619.79,194102.40,1.16,167960.72,-3320.21,98.06
4,5379.73,249039.33,225708.25,17312.19,266351.51,1.21,219629.63,-6078.63,97.31
5,5512.61,315190.60,278854.51,27461.52,342652.12,1.27,269245.30,-9609.21,96.55
6,5648.77,382975.80,330749.85,40204.67,423180.48,1.34,316868.65,-13881.20,95.80
7,5788.29,452435.31,381423.71,55684.45,508119.76,1.40,362558.98,-18864.73,95.05
8,5931.26,523610.46,430904.85,74049.84,597660.30,1.47,406374.01,-24530.84,94.31
9,6077.76,596543.64,479221.34,95456.23,691999.87,1.54,448369.93,-30851.41,93.56


In [13]:
savings_full.head(10)

year,monthly_contribution,total_contributions,real_contributions,interest_earned,nominal_corpus,inflation_factor,real_corpus,real_wealth_created,purchasing_power_retained
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,5000.00,60000.00,58458.81,1086.71,61086.71,1.05,58211.08,-247.73,99.58
2,5247.00,122964.00,116917.62,4280.02,127244.02,1.10,115546.09,-1371.52,98.83
3,5506.20,189038.42,175376.42,9753.01,198791.43,1.16,172018.23,-3358.19,98.09
4,5778.21,258376.92,233835.23,17689.60,276066.52,1.21,227640.48,-6194.75,97.35
5,6063.65,331140.74,292294.04,28285.19,359425.93,1.27,282425.63,-9868.41,96.62
6,6363.20,407499.09,350752.85,41747.33,449246.42,1.34,336386.27,-14366.57,95.90
7,6677.54,487629.55,409211.66,58296.37,545925.91,1.40,389534.83,-19676.83,95.19
8,7007.41,571718.45,467670.46,78166.19,649884.64,1.47,441883.51,-25786.96,94.49
9,7353.57,659961.34,526129.27,101604.97,761566.30,1.54,493444.35,-32684.92,93.79


In [14]:
rd

year,monthly_contribution,total_contributions,real_contributions,interest_earned,nominal_corpus,inflation_factor,real_corpus,real_wealth_created,purchasing_power_retained
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,5000.00,60000.00,58458.81,1935.65,61935.65,1.05,59020.06,561.25,100.96
2,5000.00,120000.00,114165.70,7593.29,127593.29,1.10,115863.25,1697.55,101.49
3,5000.00,180000.00,167250.21,17196.57,197196.57,1.16,170638.16,3387.96,102.03
4,5000.00,240000.00,217835.79,30982.61,270982.61,1.21,223448.37,5612.57,102.58
5,5000.00,300000.00,266040.09,49202.77,349202.77,1.27,274392.59,8352.51,103.14
6,5000.00,360000.00,311975.19,72123.52,432123.52,1.34,323565.01,11589.82,103.71
7,5000.00,420000.00,355747.91,100027.34,520027.34,1.40,371055.40,15307.49,104.30
8,5000.00,480000.00,397460.06,133213.67,613213.67,1.47,416949.40,19489.34,104.90
9,5000.00,540000.00,437208.63,171999.97,711999.97,1.54,461328.66,24120.03,105.52


In [17]:
nifty_50_fixed.head(10)

year,monthly_contribution,total_contributions,real_contributions,interest_earned,nominal_corpus,inflation_factor,real_corpus,real_wealth_created,purchasing_power_retained
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,5000.00,60000.00,58458.81,3994.20,63994.20,1.05,60981.71,2522.90,104.32
2,5000.00,120000.00,114165.70,15997.45,135997.45,1.10,123494.79,9329.10,108.17
3,5000.00,180000.00,167250.21,37012.09,217012.09,1.16,187784.94,20534.73,112.28
4,5000.00,240000.00,217835.79,68165.94,308165.94,1.21,254109.20,36273.41,116.65
5,5000.00,300000.00,266040.09,110727.93,410727.93,1.27,322737.13,56697.05,121.31
6,5000.00,360000.00,311975.19,166125.83,526125.83,1.34,393951.96,81976.77,126.28
7,5000.00,420000.00,355747.91,235966.08,655966.08,1.40,468051.85,112303.93,131.57
8,5000.00,480000.00,397460.06,322056.18,802056.18,1.47,545351.25,147891.19,137.21
9,5000.00,540000.00,437208.63,426429.86,966429.86,1.54,626182.32,188973.69,143.22


In [18]:
nifty_50_half.head(10)

year,monthly_contribution,total_contributions,real_contributions,interest_earned,nominal_corpus,inflation_factor,real_corpus,real_wealth_created,purchasing_power_retained
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,5000.00,60000.00,58458.81,3994.20,63994.20,1.05,60981.71,2522.90,104.32
2,5123.50,121482.00,115541.66,16096.11,137578.11,1.10,124930.13,9388.48,108.13
3,5250.05,184482.61,171280.93,37508.32,221990.93,1.16,192093.23,20812.30,112.15
4,5379.73,249039.33,225708.25,69588.62,318627.95,1.21,262736.03,37027.77,116.41
5,5512.61,315190.60,278854.51,113869.45,429060.05,1.27,337141.94,58287.43,120.90
6,5648.77,382975.80,330749.85,172079.93,555055.73,1.34,415614.06,84864.22,125.66
7,5788.29,452435.31,381423.71,246170.55,698605.86,1.40,498476.63,117052.92,130.69
8,5931.26,523610.46,430904.85,338341.07,861951.53,1.47,586076.58,155171.73,136.01
9,6077.76,596543.64,479221.34,451071.77,1047615.41,1.54,678785.16,199563.82,141.64


In [19]:
nifty_50_full.head(10)

year,monthly_contribution,total_contributions,real_contributions,interest_earned,nominal_corpus,inflation_factor,real_corpus,real_wealth_created,purchasing_power_retained
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,5000.00,60000.00,58458.81,3994.20,63994.20,1.05,60981.71,2522.90,104.32
2,5247.00,122964.00,116917.62,16194.76,139158.76,1.10,126365.48,9447.86,108.08
3,5506.20,189038.42,175376.42,38009.43,227047.85,1.16,196469.08,21092.66,112.03
4,5778.21,258376.92,233835.23,71040.94,329417.86,1.21,271633.23,37798.00,116.16
5,6063.65,331140.74,292294.04,117112.35,448253.09,1.27,352223.23,59929.19,120.50
6,6363.20,407499.09,350752.85,178295.60,585794.69,1.34,438630.75,87877.90,125.05
7,6677.54,487629.55,409211.66,256943.65,744573.20,1.40,531275.74,122064.08,129.83
8,7007.41,571718.45,467670.46,355726.78,927445.22,1.47,630608.46,162937.99,134.84
9,7353.57,659961.34,526129.27,477673.35,1137634.69,1.54,737111.67,210982.40,140.10


In [20]:
def push_table(df: pl.DataFrame, table_name: str) -> None:
    df.write_database(
        table_name=f"benchmark.{table_name}",
        connection=engine,
        if_table_exists="replace",
    )

In [21]:
push_table(savings_fixed, "savings_fixed")
push_table(savings_half, "savings_half")
push_table(savings_full, "savings_full")
push_table(rd, "recurring_deposit")
push_table(nifty_50_fixed, "nifty_50_fixed")
push_table(nifty_50_half, "nifty_50_half")
push_table(nifty_50_full, "nifty_50_full")
push_table(summary, "summary")

In [23]:
pl.read_database(
        query = """
        select table_name, column_name
        from information_schema.columns
        where table_schema = 'benchmark'
        order by table_name, ordinal_position;
        """, connection = engine
        )

table_name,column_name
str,str
"""nifty_50_fixed""","""year"""
"""nifty_50_fixed""","""monthly_contribution"""
"""nifty_50_fixed""","""total_contributions"""
"""nifty_50_fixed""","""real_contributions"""
"""nifty_50_fixed""","""interest_earned"""
"""nifty_50_fixed""","""nominal_corpus"""
"""nifty_50_fixed""","""inflation_factor"""
"""nifty_50_fixed""","""real_corpus"""
"""nifty_50_fixed""","""real_wealth_created"""


In [19]:
pl.read_database(
        query = """
        select *
        from benchmark.recurring_deposit;
        """, connection = engine
        )

year,monthly_contribution,total_contributions,real_contributions,interest_earned,nominal_corpus,inflation_factor,real_corpus,real_wealth_created,purchasing_power_retained
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,5000.00,60000.00,58458.81,1935.65,61935.65,1.05,59020.06,561.25,100.96
2,5000.00,120000.00,114165.70,7593.29,127593.29,1.10,115863.25,1697.55,101.49
3,5000.00,180000.00,167250.21,17196.57,197196.57,1.16,170638.16,3387.96,102.03
4,5000.00,240000.00,217835.79,30982.61,270982.61,1.21,223448.37,5612.57,102.58
5,5000.00,300000.00,266040.09,49202.77,349202.77,1.27,274392.59,8352.51,103.14
6,5000.00,360000.00,311975.19,72123.52,432123.52,1.34,323565.01,11589.82,103.71
7,5000.00,420000.00,355747.91,100027.34,520027.34,1.40,371055.40,15307.49,104.30
8,5000.00,480000.00,397460.06,133213.67,613213.67,1.47,416949.40,19489.34,104.90
9,5000.00,540000.00,437208.63,171999.97,711999.97,1.54,461328.66,24120.03,105.52
